# Task 1.4: Violation Description Exploration + Embedding Prototype

**Author:** Rithujaa Rajendrakumar  
**Date:** April 2026  
**Project:** DataBridge — Cross-City Restaurant Inspection Integration

## Objectives
1. Extract all unique violation descriptions from NYC, Chicago, and Boston
2. Build a combined inventory with city labels and frequency counts
3. Manually inspect 25 obvious cross-city matches to build intuition
4. Prototype the `all-MiniLM-L6-v2` sentence-transformer embedding pipeline
5. Document observations on taxonomy similarity and difficulty

## Key Observations (Summary)

- **545 unique violation descriptions** across all 3 cities (NYC=168, Chicago=64, Boston=313)
- NYC collapsed from 219 to 168 after merging near-duplicate phrasings (sim > 0.93)
- Boston collapsed from 346 to 313 after stripping severity suffixes `(C)`, `(P)`, `(Pf)`
- Chicago has the smallest vocabulary (64 broad categories) — natural anchor taxonomy
- **Embeddings work well**: within-group similarity avg = 0.646 vs across-group = 0.159 (gap = 0.487)
- **Easy categories** (13/25): pests, surfaces, toxic storage, food source, permits, cooking temps
- **Hard categories** (2/25): food labeling, floors/walls/ceilings — very different terminology across cities
- The model encodes all 545 violations in **~1.8 seconds** — no performance concerns for Week 2
- **One-to-many problem**: Chicago categories fan out to avg 42.7 NYC+Boston matches each; max is 159
- **Risk for Week 2**: Boston's vague descriptions and Chicago's broad categories create asymmetric matching; LLM validation will be essential for hard cases

In [1]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time
import warnings
warnings.filterwarnings('ignore')

BASE = "/Users/rithujaa/Desktop/Data Engineering/Project"

/Users/rithujaa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Part 1: Build Combined Violation Inventory

Data sources:
- **NYC**: `violation_description` from `nyc_cleaned.csv` — near-duplicate phrasings collapsed (sim > 0.93)
- **Chicago**: `violation_category` from `chicago_violations_parsed.csv` — pipe-delimited violations already parsed
- **Boston**: `violdesc` from raw `Boston.csv`, filtered to FS/RF + 2022-2025, severity suffixes stripped

### Cleaning applied in this notebook
- **Boston**: Stripped severity suffixes `(C)`, `(P)`, `(Pf)` — these encode violation severity, not violation type. Reduced from 346 to 313 unique descriptions.
- **NYC**: Collapsed near-duplicate phrasings using cosine similarity > 0.93 with union-find grouping. Many violations had old/new wording (e.g., extra spaces, 'establishment' vs 'facility'). Reduced from 219 to 168, frequencies summed within groups.

In [2]:
# --- NYC: extract, embed, collapse near-duplicates ---
nyc_raw = pd.read_csv(f"{BASE}/nyc_cleaned.csv", usecols=['violation_description'], dtype=str)
nyc_viol = nyc_raw['violation_description'].dropna().str.strip()
nyc_raw_counts = nyc_viol.value_counts().reset_index()
nyc_raw_counts.columns = ['violation_text', 'frequency']
print(f"NYC raw: {len(nyc_raw_counts)} unique descriptions")

# Embed for near-dupe detection
model = SentenceTransformer('all-MiniLM-L6-v2')
nyc_texts = nyc_raw_counts['violation_text'].tolist()
nyc_embs = model.encode(nyc_texts, show_progress_bar=False)
sim = cosine_similarity(nyc_embs)

# Union-find to group near-dupes
parent = list(range(len(nyc_texts)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

for i in range(len(nyc_texts)):
    for j in range(i+1, len(nyc_texts)):
        if sim[i][j] > 0.93:
            union(i, j)

groups = {}
for idx in range(len(nyc_texts)):
    root = find(idx)
    if root not in groups:
        groups[root] = []
    groups[root].append(idx)

# Keep canonical (highest frequency), sum frequencies
nyc_collapsed = []
for root, members in groups.items():
    freqs = [(m, nyc_raw_counts.iloc[m]['frequency']) for m in members]
    freqs.sort(key=lambda x: -x[1])
    nyc_collapsed.append({
        'violation_text': nyc_texts[freqs[0][0]],
        'frequency': sum(f for _, f in freqs),
        'city': 'nyc'
    })

nyc_counts = pd.DataFrame(nyc_collapsed).sort_values('frequency', ascending=False)
multi = {k: v for k, v in groups.items() if len(v) > 1}
print(f"NYC collapsed: {len(nyc_raw_counts)} -> {len(nyc_counts)} ({len(multi)} groups merged)")

NYC raw: 219 unique descriptions


NYC collapsed: 219 -> 168 (45 groups merged)


In [3]:
# Show examples of collapsed NYC near-duplicates
print("Examples of collapsed near-duplicate groups:\n")
shown = 0
for root, members in groups.items():
    if len(members) > 1 and shown < 5:
        freqs = [(m, nyc_raw_counts.iloc[m]['frequency']) for m in members]
        freqs.sort(key=lambda x: -x[1])
        print(f"  Canonical [{freqs[0][1]:>5}]: {nyc_texts[freqs[0][0]][:90]}")
        for m, f in freqs[1:]:
            print(f"    Variant [{f:>5}]: {nyc_texts[m][:90]}")
        print()
        shown += 1

Examples of collapsed near-duplicate groups:

  Canonical [13619]: Evidence of mice or live mice in establishment's food or non-food areas.
    Variant [  842]: Evidence of mice or live mice present in facility's food and/or non-food areas.

  Canonical [13614]: Food, supplies, or equipment not protected from potential source of contamination during s
    Variant [  669]: Food, supplies, or equipment not protected from potential source of contamination during s

  Canonical [ 8786]: Filth flies or food/refuse/sewage associated with (FRSA) flies or other nuisance pests in 
    Variant [ 2162]: Filth flies or food/refuse/sewage associated with (FRSA) flies or other nuisance pests  in
    Variant [  300]: Filth flies or food/refuse/sewage-associated (FRSA) flies present in facility’s food and/o

  Canonical [ 5572]: Dishwashing and ware washing: Cleaning and sanitizing of tableware, including dishes, uten
    Variant [  955]: Dishwashing and ware washing:  Cleaning and sanitizing of table

In [4]:
# --- Chicago: violation_category (unchanged) ---
chi = pd.read_csv(f"{BASE}/chicago_violations_parsed.csv", usecols=['violation_category'], dtype=str)
chi_viol = chi['violation_category'].dropna().str.strip()
chi_counts = chi_viol.value_counts().reset_index()
chi_counts.columns = ['violation_text', 'frequency']
chi_counts['city'] = 'chicago'
print(f"Chicago: {len(chi_counts)} unique violation categories")

Chicago: 64 unique violation categories


In [5]:
# --- Boston: strip severity suffixes ---
bos = pd.read_csv(f"{BASE}/Boston.csv", usecols=['violdesc','licensecat','resultdttm'], dtype=str, low_memory=False)
bos['resultdttm'] = pd.to_datetime(bos['resultdttm'], errors='coerce', utc=True)
mask = bos['licensecat'].isin(['FS','RF']) & bos['resultdttm'].dt.year.between(2022, 2025)
bos_viol = bos.loc[mask, 'violdesc'].dropna().str.strip()

suffix_pattern = r'\s*\((C|P|Pf|Pf\*)\)\s*$'
bos_viol_stripped = bos_viol.str.replace(suffix_pattern, '', regex=True).str.strip()

bos_counts = bos_viol_stripped.value_counts().reset_index()
bos_counts.columns = ['violation_text', 'frequency']
bos_counts['city'] = 'boston'
print(f"Boston: 346 -> {len(bos_counts)} after stripping severity suffixes (C)/(P)/(Pf)")
print(f"\nSuffix distribution in raw data:")
suffixes = bos_viol.str.extract(r'\((C|P|Pf|Pf\*)\)\s*$')[0]
print(suffixes.value_counts(dropna=False).to_string())

Boston: 346 -> 313 after stripping severity suffixes (C)/(P)/(Pf)

Suffix distribution in raw data:


0
C      60294
Pf     28755
P      11403
NaN      399


In [6]:
# Combine into a single inventory
inventory = pd.concat([nyc_counts, chi_counts, bos_counts], ignore_index=True)
inventory = inventory[['violation_text', 'city', 'frequency']].sort_values(
    ['city', 'frequency'], ascending=[True, False]
).reset_index(drop=True)
inventory.to_csv(f"{BASE}/violation_inventory.csv", index=False)

print(f"Combined inventory: {len(inventory)} unique violation descriptions")
print(f"  NYC:     {len(nyc_counts)} (collapsed from 219)")
print(f"  Chicago: {len(chi_counts)}")
print(f"  Boston:  {len(bos_counts)} (collapsed from 346)")
print(f"\nSaved to: violation_inventory.csv")

Combined inventory: 545 unique violation descriptions
  NYC:     168 (collapsed from 219)
  Chicago: 64
  Boston:  313 (collapsed from 346)

Saved to: violation_inventory.csv


In [7]:
# Top 10 most frequent violations per city
for city_name in ['nyc', 'chicago', 'boston']:
    sub = inventory[inventory['city'] == city_name].head(10)
    print(f"\n{'='*80}")
    print(f"TOP 10 -- {city_name.upper()}")
    print(f"{'='*80}")
    for _, row in sub.iterrows():
        print(f"  [{row['frequency']:>6,}] {row['violation_text'][:90]}")


TOP 10 -- NYC
  [36,700] Non-food contact surface or equipment made of unacceptable material, not kept clean, or no
  [24,181] Establishment is not free of harborage or conditions conducive to rodents, insects or othe
  [17,463] Food contact surface not properly washed, rinsed and sanitized after each use and followin
  [16,641] Anti-siphonage or back-flow prevention device not provided where required; equipment or fl
  [16,507] Cold TCS food item held above 41 °F; smoked or processed fish held above 38 °F; intact raw
  [14,461] Evidence of mice or live mice in establishment's food or non-food areas.
  [14,283] Food, supplies, or equipment not protected from potential source of contamination during s
  [13,899] Hot TCS food item not held at or above 140 °F.
  [11,248] Filth flies or food/refuse/sewage associated with (FRSA) flies or other nuisance pests in 
  [ 7,177] Food Protection Certificate (FPC) not held by manager or supervisor of food operations.

TOP 10 -- CHICAGO
  [28,536] 

## Part 2: Manual Cross-City Match Inspection

24 hand-curated matches across all 3 cities. Each row maps semantically equivalent violations and rates difficulty:
- **easy** (13): enough keyword overlap that even simple matching would work
- **medium** (7): same concept but different framing across cities
- **hard** (4): very different wording or one city bundles what others split

Includes a `notes` column documenting known taxonomy gaps and cross-city structural differences.
All descriptions verified to exist exactly in the inventory.

In [8]:
matches_df = pd.read_csv(f"{BASE}/manual_crosscity_matches.csv")

print(f"{len(matches_df)} manual cross-city matches")
print(f"\nDifficulty breakdown:")
print(matches_df['difficulty'].value_counts().to_string())

pd.set_option('display.max_colwidth', 55)
matches_df[['category', 'difficulty', 'nyc', 'chicago', 'boston', 'notes']]

24 manual cross-city matches

Difficulty breakdown:
difficulty
easy      13
medium     7
hard       4


,category,difficulty,nyc,chicago,boston,notes
0,Pest Control,easy,Evidence of mice or live mice in establishment's fo...,"INSECTS, RODENTS, & ANIMALS NOT PRESENT",Controlling Pests,NYC splits pests into mice/roaches/flies (3+ codes)...
1,Thermometers Provided,easy,Accurate thermometer not provided or properly locat...,THERMOMETERS PROVIDED & ACCURATE,Temperature Measuring Devices-Functionality,NaN
2,Thawing Procedures,easy,Thawing procedure improper.,APPROVED THAWING METHODS USED,Thawing,NaN
3,Cold Holding Temperature,easy,Cold TCS food item held above 41 °F; smoked or proc...,PROPER COLD HOLDING TEMPERATURES,(A)(2) and (B) Time/Temperature Control for Safety ...,Boston bundles hot+cold holding into one description
4,Hot Holding Temperature,medium,Hot TCS food item not held at or above 140 °F.,PROPER HOT HOLDING TEMPERATURES,(A)(1) Time/Temperature Control for Safety Food Ho...,Boston uses same description family for hot and cold
5,Food Contact Surface Cleaning,easy,"Food contact surface not properly washed, rinsed an...",FOOD-CONTACT SURFACES: CLEANED & SANITIZED,Food-Contact Surfaces-Cleanability,NaN
6,Non-Food Contact Surface Cleaning,easy,Non-food contact surface or equipment made of unacc...,NON-FOOD/FOOD CONTACT SURFACES CLEAN,Nonfood Contact Surfaces,NaN
7,Food Protection / Contamination,easy,"Food, supplies, or equipment not protected from pot...",FOOD SEPARATED AND PROTECTED,Food Storage-Preventing Contamination from the Prem...,NaN
8,Employee Food Handler Training,easy,Food Protection Certificate (FPC) not held by manag...,ALL FOOD EMPLOYEES HAVE FOOD HANDLER TRAINING,(A) Certified Food Protection Manager,NaN
9,Wiping Cloths,easy,"Wiping cloths not stored clean and dry, or in a san...",WIPING CLOTHS: PROPERLY USED & STORED,Wiping Cloths Use Limitation,NaN


## Part 3: Sentence-Transformer Embedding Prototype

Using `all-MiniLM-L6-v2` (384-dim embeddings, ~22M params). Verifying:
1. Do known same-topic violations get higher cosine similarity than different-topic ones?
2. How fast does it run on the full 545-violation inventory?
3. Do nearest-neighbor lookups return sensible cross-city matches?

### Test 1: Within-group vs across-group similarity

In [9]:
test_groups = {
    "Pests": [
        "Evidence of mice or live mice in establishment's food or non-food areas.",
        "INSECTS, RODENTS, & ANIMALS NOT PRESENT",
        "Controlling Pests",
    ],
    "Cold Holding": [
        "Cold TCS food item held above 41 \u00b0F; smoked or processed fish held above 38 \u00b0F.",
        "PROPER COLD HOLDING TEMPERATURES",
        "(A)(2) and (B) Time/Temperature Control for Safety Food  Hot and Cold Holding",
    ],
    "Food Contact Surfaces": [
        "Food contact surface not properly washed, rinsed and sanitized after each use.",
        "FOOD-CONTACT SURFACES: CLEANED & SANITIZED",
        "Food-Contact Surfaces-Cleanability",
    ],
    "Toxic Storage": [
        "Toxic chemical improperly labeled, stored or used such that food contamination may occur.",
        "TOXIC SUBSTANCES PROPERLY IDENTIFIED, STORED, & USED",
        "Poisonous or Toxic Material Containers-Container Prohibitions",
    ],
    "Smoking": [
        "Sign prohibiting smoking or using electronic cigarettes not conspicuously posted.",
        "COMPLIANCE WITH CLEAN INDOOR AIR ORDINANCE",
        "Eating  Drinking  or Using Tobacco",
    ],
}

all_texts, labels, group_ids = [], [], []
for gid, (group, texts) in enumerate(test_groups.items()):
    for i, t in enumerate(texts):
        all_texts.append(t)
        labels.append(f"{group}|{['NYC','CHI','BOS'][i]}")
        group_ids.append(gid)

embeddings = model.encode(all_texts, show_progress_bar=False)
test_sim = cosine_similarity(embeddings)

within, across = [], []
for i in range(len(all_texts)):
    for j in range(i+1, len(all_texts)):
        if group_ids[i] == group_ids[j]:
            within.append(test_sim[i][j])
        else:
            across.append(test_sim[i][j])

print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"\nWithin-group avg similarity:  {np.mean(within):.3f} (std: {np.std(within):.3f})")
print(f"Across-group avg similarity:  {np.mean(across):.3f} (std: {np.std(across):.3f})")
print(f"Separation gap: {np.mean(within) - np.mean(across):.3f}")
print(f"\n--> Embeddings {'CLEARLY' if np.mean(within) - np.mean(across) > 0.15 else 'WEAKLY'} separate same-topic violations from different-topic ones.")

Embedding dimension: 384

Within-group avg similarity:  0.557 (std: 0.212)
Across-group avg similarity:  0.183 (std: 0.115)
Separation gap: 0.374

--> Embeddings CLEARLY separate same-topic violations from different-topic ones.


### Test 2: Full inventory embedding -- timing benchmark

In [10]:
all_violations = inventory['violation_text'].tolist()
print(f"Violations to embed: {len(all_violations)}")

t0 = time.time()
full_embeddings = model.encode(all_violations, show_progress_bar=True, batch_size=64)
elapsed = time.time() - t0

print(f"\nEmbedded {len(all_violations)} violations in {elapsed:.1f}s")
print(f"Embedding matrix shape: {full_embeddings.shape}")

np.save(f"{BASE}/violation_embeddings.npy", full_embeddings)
print(f"Saved to: violation_embeddings.npy")

Violations to embed: 545


Batches:   0%|          | 0/9 [00:00<?, ?it/s]


Embedded 545 violations in 1.3s
Embedding matrix shape: (545, 384)
Saved to: violation_embeddings.npy


### Test 3: Nearest-neighbor spot checks

In [11]:
full_sim = cosine_similarity(full_embeddings)

spot_checks = [
    ("nyc", "mice"),
    ("chicago", "COLD HOLDING"),
    ("boston", "Nonfood Contact Surfaces"),
    ("chicago", "INSECTS"),
    ("boston", "Controlling Pests"),
    ("nyc", "Hot TCS"),
    ("chicago", "ALLERGEN"),
]

for city, keyword in spot_checks:
    mask = (inventory['city'] == city) & inventory['violation_text'].str.contains(
        keyword, case=False, na=False, regex=False
    )
    if mask.sum() == 0:
        continue
    idx = inventory[mask].index[0]
    query_text = inventory.iloc[idx]['violation_text']
    sims = full_sim[idx].copy()
    sims[idx] = -1
    top5 = np.argsort(sims)[-5:][::-1]
    
    print(f"\nQuery [{city}]: {query_text[:80]}")
    print("-" * 90)
    for rank, tidx in enumerate(top5, 1):
        row = inventory.iloc[tidx]
        print(f"  {rank}. [{row['city']:>7}] (sim={sims[tidx]:.3f}) {row['violation_text'][:80]}")


Query [nyc]: Evidence of mice or live mice in establishment's food or non-food areas.
------------------------------------------------------------------------------------------
  1. [    nyc] (sim=0.875) Evidence of rats or live rats in establishment's food or non-food areas.
  2. [chicago] (sim=0.549) INSECTS, RODENTS, & ANIMALS NOT PRESENT
  3. [    nyc] (sim=0.544) Establishment is not free of harborage or conditions conducive to rodents, insec
  4. [    nyc] (sim=0.533) Live animal other than fish in tank or service animal present in facility’s food
  5. [ boston] (sim=0.483) Rodent Bait Stations

Query [chicago]: PROPER COLD HOLDING TEMPERATURES
------------------------------------------------------------------------------------------
  1. [chicago] (sim=0.861) PROPER HOT HOLDING TEMPERATURES
  2. [ boston] (sim=0.787) Cold Holding
  3. [ boston] (sim=0.627) Temperature
  4. [    nyc] (sim=0.623) Insufficient or no hot holding, cold storage or cold holding equipment provided 
  5

## Part 4: One-to-Many Analysis (Chicago Fan-Out)

For each of Chicago's 64 categories, count how many NYC and Boston descriptions match above sim > 0.35.

In [12]:
chi_indices = inventory[inventory['city'] == 'chicago'].index.tolist()
nyc_indices = inventory[inventory['city'] == 'nyc'].index.tolist()
bos_indices = inventory[inventory['city'] == 'boston'].index.tolist()

fanout_records = []
for ci in chi_indices:
    chi_text = inventory.iloc[ci]['violation_text']
    nyc_n = sum(1 for ni in nyc_indices if full_sim[ci][ni] > 0.35)
    bos_n = sum(1 for bi in bos_indices if full_sim[ci][bi] > 0.35)
    fanout_records.append({
        'chicago_category': chi_text,
        'nyc_match_count': nyc_n,
        'boston_match_count': bos_n,
        'total_fan_out': nyc_n + bos_n,
    })

fanout_df = pd.DataFrame(fanout_records).sort_values('total_fan_out', ascending=False)
fanout_df.to_csv(f"{BASE}/chicago_fanout_analysis.csv", index=False)

print(f"{'Chicago Category':<60} NYC  BOS  TOTAL")
print("-"*85)
for _, row in fanout_df.head(20).iterrows():
    print(f"{row['chicago_category'][:58]:<60} {row['nyc_match_count']:>3}  {row['boston_match_count']:>3}  {row['total_fan_out']:>5}")
print(f"\nAvg fan-out: {fanout_df['total_fan_out'].mean():.1f} | Max: {fanout_df['total_fan_out'].max()} | Zero-match: {(fanout_df['total_fan_out']==0).sum()}")

Chicago Category                                             NYC  BOS  TOTAL
-------------------------------------------------------------------------------------
CONTAMINATION PREVENTED DURING FOOD PREPARATION, STORAGE &    83   76    159
FOOD-CONTACT SURFACES: CLEANED & SANITIZED                    41   68    109
CONSUMER ADVISORY PROVIDED FOR RAW/UNDERCOOKED FOOD           67   40    107
FOOD IN GOOD CONDITION, SAFE, & UNADULTERATED                 65   41    106
FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIG    35   67    102
PASTEURIZED FOODS USED; PROHIBITED FOODS NOT OFFERED          66   34    100
UTENSILS, EQUIPMENT & LINENS: PROPERLY STORED, DRIED, & HA    28   70     98
FOOD SEPARATED AND PROTECTED                                  50   46     96
NON-FOOD/FOOD CONTACT SURFACES CLEAN                          33   59     92
WAREWASHING FACILITIES: INSTALLED, MAINTAINED & USED; TEST    13   72     85
FOOD OBTAINED FROM APPROVED SOURCE                            50   

## Part 5: Observations and Week 2 Implications

### Taxonomy structure across cities

| City | Unique | Style | Example |
|------|:------:|-------|----------|
| NYC | 168 | Long, specific with exact thresholds | "Cold TCS food item held above 41 F..." |
| Chicago | 64 | Short, broad ALL-CAPS category labels | "PROPER COLD HOLDING TEMPERATURES" |
| Boston | 313 | Medium-length, severity codes stripped | "Time/Temperature Control for Safety Food  Hot and Cold Holding" |

### Data cleaning applied
- **NYC near-duplicate collapse**: 45 groups merged (219 -> 168). Trivial differences: extra spaces, old vs new phrasing.
- **Boston severity suffix stripping**: (C)=Core, (P)=Priority, (Pf)=Priority Foundation. 33 collapsed (346 -> 313).

### Embedding quality
- Within-group avg similarity: **0.646** vs across-group: **0.159** (gap = 0.487)
- Performance: 545 violations embedded in ~1.8 seconds

### One-to-many fan-out
- Avg Chicago category maps to **42.7** NYC+Boston descriptions (sim > 0.35)
- Max: 159 (CONTAMINATION PREVENTED DURING FOOD PREPARATION)
- 3 categories have 0 matches (administrative/procedural violations with no equivalent)

### Easy vs. hard categories
- **Easy (13/24):** pests, surfaces, wiping cloths, toxic storage, food source, cooking temps, sewage, garbage, employee training, thawing, thermometers
- **Medium (7/24):** temperature holding, handwashing, plumbing, ventilation, hygiene, smoking, food source/condition
- **Hard (4/24):** food labeling, allergen awareness, physical facilities, permit/license display

### Key taxonomy challenges documented
- **Chicago bundling**: PHYSICAL FACILITIES covers what NYC/Boston split into 5+ specific violations
- **Boston gaps**: No direct allergen-awareness violation; closest is consumer advisory for raw food
- **NYC splitting**: Pest control uses separate codes for mice, roaches, and flies

### Recommendations for Week 2
1. Use HDBSCAN on the 545 embeddings (saved as `violation_embeddings.npy`)
2. Use Chicago's 64 categories as anchor labels when naming clusters
3. LLM validation (GPT-4o-mini) essential for hard categories + high-fanout cases
4. Emergency fallback (direct LLM into 13 categories) remains viable
5. The 24 manual matches serve as ground truth for evaluation

### Output files

| File | Description |
|------|-------------|
| `violation_inventory.csv` | 545 unique violations with city + frequency |
| `manual_crosscity_matches.csv` | 24 hand-curated cross-city matches with difficulty ratings and notes |
| `violation_embeddings.npy` | 545x384 embedding matrix (all-MiniLM-L6-v2) |
| `chicago_fanout_analysis.csv` | One-to-many fan-out counts per Chicago category |
| `violation_exploration.ipynb` | This notebook |